# sphinxcontrib_jquery

sphinxcontrib-jquery 4.1 puts jQuery 3.6.0 back into Sphinx sites (Sphinx stopped shipping it in 6.0), with Sphinx's old
jQuery helpers beside it. python/sphinx_rtd_theme needs it and switches it on itself. These cells build small projects in
a temporary folder: the two files and their script tags, the integrity hashes, a rebuild, a strict parallel build, and the
extension used through the theme without a duplicate tag.

In [ ]:
# Example 1: with the extension on, jquery.js and the helpers land in _static and every page loads them.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.jquery']\n")
(src / "index.rst").write_text("Title\n=====\n\n.. toctree::\n\n   other\n")
(src / "other.rst").write_text("Other\n=====\n\nText.\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in (out / "_static").glob("*j*.js")))
assert rc == 0
for page in ("index.html", "other.html"):
    html = (out / page).read_text()
    assert 'src="_static/jquery.js' in html and 'src="_static/_sphinx_javascript_frameworks_compat.js' in html, page
assert (out / "_static" / "jquery.js").read_text().startswith("/*! jQuery v3.6.0 |")

In [ ]:
# Example 2: jquery_use_sri adds integrity hashes, and each hash is the sha384 of the file the site serves.
import base64, hashlib, pathlib, re, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.jquery']\njquery_use_sri = True\n")
(src / "index.rst").write_text("Title\n=====\n\nText.\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
html = (src / "_out" / "index.html").read_text()
tags = dict(re.findall(r'<script integrity="(sha384-[^"]+)" src="_static/([^"?]+)', html))
print("rc", rc, sorted(tags.values()))
assert rc == 0 and sorted(tags.values()) == ["_sphinx_javascript_frameworks_compat.js", "jquery.js"]
for digest, name in tags.items():
    data = (src / "_out" / "_static" / name).read_bytes()
    assert digest == "sha384-" + base64.b64encode(hashlib.sha384(data).digest()).decode(), name

In [ ]:
# Example 3: the served files are the package's own copies, byte for byte, and they survive a rebuild.
import filecmp, pathlib, tempfile
import sphinxcontrib.jquery as jq
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.jquery']\n")
(src / "index.rst").write_text("Title\n=====\n\nText.\n")
for attempt in (1, 2):  # the second build reuses the first one's environment
    assert build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")]) == 0, attempt
static = src / "_out" / "_static"
for name, _integrity in jq._FILES:
    assert filecmp.cmp(static / name, pathlib.Path(jq._ROOT_DIR) / name, shallow=False), name
print(jq.__version__, [name for name, _ in jq._FILES])
assert jq.__version__ == "4.1" and jq.version_info == (4, 1)

In [ ]:
# Example 4: the extension declares itself parallel-safe and survives a strict two-worker build.
import pathlib, tempfile
from unittest import mock
from sphinx.cmd.build import build_main
import sphinxcontrib.jquery as jq

info = jq.setup(mock.MagicMock())  # the entry point Sphinx calls for an extension
assert info["parallel_read_safe"] is True and info["parallel_write_safe"] is True, info
src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.jquery']\n")
pages = "\n".join(f"   page{i}" for i in range(6))
(src / "index.rst").write_text(f"Title\n=====\n\n.. toctree::\n\n{pages}\n")
for i in range(6):
    (src / f"page{i}.rst").write_text(f"Page {i}\n=======\n\nText {i}.\n")
rc = build_main(["-q", "-W", "-j", "2", "-b", "html", str(src), str(src / "_out")])
print("rc", rc)
assert rc == 0
assert 'src="_static/jquery.js' in (src / "_out" / "page5.html").read_text()

In [ ]:
# Example 5: through python/sphinx_rtd_theme: listed by hand as well, each script tag still appears once.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.jquery']\nhtml_theme = 'sphinx_rtd_theme'\n")
(src / "index.rst").write_text("Title\n=====\n\nText.\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
html = (src / "_out" / "index.html").read_text()
counts = {name: html.count(f'src="_static/{name}') for name in ("jquery.js", "_sphinx_javascript_frameworks_compat.js")}
print("rc", rc, counts)
assert rc == 0
assert counts == {"jquery.js": 1, "_sphinx_javascript_frameworks_compat.js": 1}, counts
assert "_static/css/theme.css" in html